<a href="https://colab.research.google.com/github/DhikshaSubash/market-sentiment-analysis/blob/main/notebooks/05_correlation_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - Setup
!pip install pyspark -q

from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, round as spark_round,
    lag, when, abs as spark_abs,
    to_timestamp, date_trunc
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("SentimentCorrelation") \
    .getOrCreate()

# Load sentiment and price data
sentiment_df = spark.read.parquet(
    "/content/drive/MyDrive/market_sentiment/sentiment/scored_news"
)
price_df = spark.read.parquet(
    "/content/drive/MyDrive/market_sentiment/processed/prices"
)

print(f"✅ Sentiment records: {sentiment_df.count()}")
print(f"✅ Price records: {price_df.count()}")

Mounted at /content/drive
✅ Sentiment records: 1288
✅ Price records: 29248


In [2]:
# Cell 2 - Calculate price movement per stock
window_spec = Window.partitionBy("symbol").orderBy("timestamp")

price_movement_df = price_df \
    .withColumn("prev_price", lag("price", 1).over(window_spec)) \
    .withColumn("price_change",
        when(col("prev_price").isNotNull(),
            spark_round(
                ((col("price") - col("prev_price")) / col("prev_price")) * 100
            , 4)
        ).otherwise(0.0)
    ) \
    .withColumn("movement_label",
        when(col("price_change") > 0.1,  "up")
        .when(col("price_change") < -0.1, "down")
        .otherwise("stable")
    )

print("✅ Price movement calculated!")
print("\nPrice movement sample:")
price_movement_df.select(
    "symbol", "price", "prev_price",
    "price_change", "movement_label", "timestamp"
).show(5)

print("\nMovement distribution:")
price_movement_df.groupBy("movement_label") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

✅ Price movement calculated!

Price movement sample:
+------+------+----------+------------+--------------+-------------------+
|symbol| price|prev_price|price_change|movement_label|          timestamp|
+------+------+----------+------------+--------------+-------------------+
|  AAPL|248.36|      NULL|         0.0|        stable|2026-03-19 13:30:00|
|  AAPL|248.56|    248.36|      0.0805|        stable|2026-03-19 13:31:00|
|  AAPL|248.13|    248.56|      -0.173|          down|2026-03-19 13:32:00|
|  AAPL|248.59|    248.13|      0.1854|            up|2026-03-19 13:33:00|
|  AAPL|248.79|    248.59|      0.0805|        stable|2026-03-19 13:34:00|
+------+------+----------+------------+--------------+-------------------+
only showing top 5 rows

Movement distribution:
+--------------+-----+
|movement_label|count|
+--------------+-----+
|        stable|23889|
|          down| 2684|
|            up| 2675|
+--------------+-----+



In [3]:
# Cell 3 - Aggregate sentiment scores per stock
from pyspark.sql.functions import expr

sentiment_agg_df = sentiment_df.groupBy("symbol").agg(
    spark_round(avg("positive_score"), 4).alias("avg_positive"),
    spark_round(avg("negative_score"), 4).alias("avg_negative"),
    spark_round(avg("neutral_score"),  4).alias("avg_neutral"),
)

# Composite sentiment score: positive - negative (ranges -1 to +1)
sentiment_agg_df = sentiment_agg_df.withColumn(
    "composite_sentiment",
    spark_round(col("avg_positive") - col("avg_negative"), 4)
)

print("✅ Sentiment aggregated per stock!")
sentiment_agg_df.orderBy("composite_sentiment", ascending=False).show()

✅ Sentiment aggregated per stock!
+------+------------+------------+-----------+-------------------+
|symbol|avg_positive|avg_negative|avg_neutral|composite_sentiment|
+------+------------+------------+-----------+-------------------+
| GOOGL|      0.2968|      0.1397|     0.5635|             0.1571|
|   AMD|      0.2582|      0.1167|     0.6251|             0.1415|
|    MA|      0.1893|      0.0621|     0.7486|             0.1272|
|  MSFT|      0.2692|      0.1517|     0.5791|             0.1175|
|  AMZN|      0.0988|      0.0392|      0.862|             0.0596|
|  NVDA|      0.2292|      0.1738|      0.597|             0.0554|
|  AAPL|       0.147|      0.1009|      0.752|             0.0461|
|    GS|      0.1859|      0.1591|      0.655|             0.0268|
|  INTC|      0.3398|      0.3331|     0.3271|             0.0067|
|  NFLX|      0.3167|       0.337|     0.3462|            -0.0203|
|   JPM|      0.1871|      0.2277|     0.5853|            -0.0406|
|     V|      0.1369|      0

In [4]:
# Cell 4 - Aggregate price stats per stock
price_agg_df = price_movement_df.groupBy("symbol").agg(
    spark_round(avg("price"), 2).alias("avg_price"),
    spark_round(avg("price_change"), 4).alias("avg_price_change"),
    spark_round(avg(spark_abs(col("price_change"))), 4).alias("avg_volatility")
)

print("✅ Price aggregated per stock!")
price_agg_df.orderBy("avg_volatility", ascending=False).show()

✅ Price aggregated per stock!
+------+---------+----------------+--------------+
|symbol|avg_price|avg_price_change|avg_volatility|
+------+---------+----------------+--------------+
|  INTC|    45.17|          0.0038|        0.1114|
|   AMD|   205.73|          0.0065|        0.0977|
|    GS|   826.86|          0.0033|        0.0756|
|  TSLA|   381.54|          5.0E-4|        0.0706|
|  NVDA|   176.83|          4.0E-4|        0.0613|
|  NFLX|    92.16|         -0.0016|        0.0593|
|  AMZN|   208.92|          0.0012|        0.0583|
|   JPM|   290.72|          0.0019|        0.0575|
|   BAC|    47.69|          0.0026|         0.057|
|  META|   599.52|         -0.0012|        0.0566|
|    MA|   498.16|          0.0013|        0.0536|
| GOOGL|   298.94|         -0.0021|        0.0534|
|     V|   303.08|          0.0011|        0.0497|
|  MSFT|   380.32|         -0.0024|        0.0483|
|  AAPL|   251.07|          9.0E-4|        0.0478|
+------+---------+----------------+--------------+



In [5]:
# Cell 5 - Join sentiment and price data
combined_df = sentiment_agg_df.join(price_agg_df, on="symbol", how="inner")

print("✅ Sentiment + Price data joined!")
print(f"Combined rows: {combined_df.count()}")
print("\nCombined feature table:")
combined_df.show()

✅ Sentiment + Price data joined!
Combined rows: 15

Combined feature table:
+------+------------+------------+-----------+-------------------+---------+----------------+--------------+
|symbol|avg_positive|avg_negative|avg_neutral|composite_sentiment|avg_price|avg_price_change|avg_volatility|
+------+------------+------------+-----------+-------------------+---------+----------------+--------------+
|  AAPL|       0.147|      0.1009|      0.752|             0.0461|   251.07|          9.0E-4|        0.0478|
|   AMD|      0.2582|      0.1167|     0.6251|             0.1415|   205.73|          0.0065|        0.0977|
|  AMZN|      0.0988|      0.0392|      0.862|             0.0596|   208.92|          0.0012|        0.0583|
|   BAC|      0.1172|      0.2633|     0.6194|            -0.1461|    47.69|          0.0026|         0.057|
| GOOGL|      0.2968|      0.1397|     0.5635|             0.1571|   298.94|         -0.0021|        0.0534|
|    GS|      0.1859|      0.1591|      0.655|      

In [6]:
# Cell 6 - Bucket prices into 30-min windows per stock
from pyspark.sql.functions import window

price_windowed_df = price_movement_df \
    .withColumn("timestamp", col("timestamp").cast("timestamp")) \
    .groupBy(
        "symbol",
        window(col("timestamp"), "30 minutes").alias("time_window")
    ).agg(
        spark_round(avg("price"), 2).alias("avg_price"),
        spark_round(avg("price_change"), 4).alias("avg_price_change"),
        spark_round(avg(spark_abs(col("price_change"))), 4).alias("avg_volatility")
    ) \
    .withColumn("window_start", col("time_window.start")) \
    .drop("time_window")

print(f"✅ Price windows created: {price_windowed_df.count()} rows")
price_windowed_df.show(5)

✅ Price windows created: 975 rows
+------+---------+----------------+--------------+-------------------+
|symbol|avg_price|avg_price_change|avg_volatility|       window_start|
+------+---------+----------------+--------------+-------------------+
|  AAPL|   250.14|          0.0324|        0.1025|2026-03-19 13:30:00|
|  AAPL|   250.47|          -0.013|        0.0631|2026-03-19 14:00:00|
|  AAPL|   249.98|           0.001|        0.0617|2026-03-19 14:30:00|
|  AAPL|    249.7|         -0.0061|        0.0464|2026-03-19 15:00:00|
|  AAPL|   249.31|          0.0025|        0.0362|2026-03-19 15:30:00|
+------+---------+----------------+--------------+-------------------+
only showing top 5 rows


In [7]:
# Cell 7 - Join with sentiment and assemble features
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Join windowed prices with sentiment
model_input_df = price_windowed_df.join(
    sentiment_agg_df, on="symbol", how="inner"
)

print(f"✅ Model input rows: {model_input_df.count()}")

# Feature assembly
feature_cols = [
    "avg_positive",
    "avg_negative",
    "avg_neutral",
    "composite_sentiment",
    "avg_volatility"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
assembled_df = assembler.transform(model_input_df)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(assembled_df)
scaled_df = scaler_model.transform(assembled_df)

# Label
model_df = scaled_df.withColumnRenamed("avg_price_change", "label")

# Train/test split
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows : {train_df.count()}")
print(f"Testing rows  : {test_df.count()}")

✅ Model input rows: 975
Training rows : 817
Testing rows  : 158


In [8]:
# Cell 8 - Train model + check loss metrics
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr = LinearRegression(
    featuresCol="scaled_features",
    labelCol="label",
    maxIter=100,
    regParam=0.1,       # L2 regularization
    elasticNetParam=0.5 # Mix of L1 + L2
)

lr_model = lr.fit(train_df)

# Predictions
train_preds = lr_model.transform(train_df)
test_preds  = lr_model.transform(test_df)

# Evaluate
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_mae  = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
evaluator_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

train_rmse = evaluator_rmse.evaluate(train_preds)
test_rmse  = evaluator_rmse.evaluate(test_preds)
train_mae  = evaluator_mae.evaluate(train_preds)
test_mae   = evaluator_mae.evaluate(test_preds)
train_r2   = evaluator_r2.evaluate(train_preds)
test_r2    = evaluator_r2.evaluate(test_preds)

print("=" * 45)
print("        📊 MODEL EVALUATION REPORT")
print("=" * 45)
print(f"  {'Metric':<10} {'Train':>10} {'Test':>10}")
print("-" * 45)
print(f"  {'RMSE':<10} {train_rmse:>10.6f} {test_rmse:>10.6f}")
print(f"  {'MAE':<10} {train_mae:>10.6f} {test_mae:>10.6f}")
print(f"  {'R²':<10} {train_r2:>10.6f} {test_r2:>10.6f}")
print("=" * 45)

# Overfitting check
if abs(train_r2 - test_r2) > 0.2:
    print("\n⚠️  WARNING: Possible overfitting detected!")
    print("   Train R² and Test R² differ significantly")
else:
    print("\n✅ Model looks stable - no major overfitting")

print(f"\nCoefficients : {lr_model.coefficients}")
print(f"Intercept    : {round(lr_model.intercept, 6)}")

        📊 MODEL EVALUATION REPORT
  Metric          Train       Test
---------------------------------------------
  RMSE         0.019054   0.018971
  MAE          0.011542   0.010553
  R²          -0.000000  -0.004328

✅ Model looks stable - no major overfitting

Coefficients : [0.0,0.0,0.0,0.0,0.0]
Intercept    : 0.001281


In [9]:
# Cell 9 - Predict Sentiment Impact Score for each stock
predictions_df = lr_model.transform(scaled_df)

impact_df = predictions_df.withColumn(
    "sentiment_impact_score",
    spark_round(col("prediction"), 6)
).select(
    "symbol",
    "avg_positive",
    "avg_negative",
    "composite_sentiment",
    "avg_price",
    "avg_price_change",
    "avg_volatility",
    "sentiment_impact_score"
).dropDuplicates(["symbol"])

print("✅ Sentiment Impact Scores generated!")
print("\n📊 Final Sentiment Impact Report:")
print("(+ve = sentiment pushing price UP)")
print("(-ve = sentiment pushing price DOWN)\n")
impact_df.orderBy("sentiment_impact_score", ascending=False).show()

✅ Sentiment Impact Scores generated!

📊 Final Sentiment Impact Report:
(+ve = sentiment pushing price UP)
(-ve = sentiment pushing price DOWN)

+------+------------+------------+-------------------+---------+----------------+--------------+----------------------+
|symbol|avg_positive|avg_negative|composite_sentiment|avg_price|avg_price_change|avg_volatility|sentiment_impact_score|
+------+------------+------------+-------------------+---------+----------------+--------------+----------------------+
|  AAPL|       0.147|      0.1009|             0.0461|   250.14|          0.0324|        0.1025|              0.001281|
|   AMD|      0.2582|      0.1167|             0.1415|   194.64|          0.0341|        0.1555|              0.001281|
|  AMZN|      0.0988|      0.0392|             0.0596|   207.71|          0.0228|         0.136|              0.001281|
|   BAC|      0.1172|      0.2633|            -0.1461|     46.6|          0.0159|        0.1202|              0.001281|
| GOOGL|      0.

In [10]:
# Cell 10 - Save correlation results
impact_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/market_sentiment/correlation/impact_scores"
)

impact_df.toPandas().to_csv(
    "/content/drive/MyDrive/market_sentiment/correlation/impact_scores.csv",
    index=False
)

print("✅ Correlation results saved!")
print("""
market_sentiment/
├── raw/           ✅
├── processed/     ✅
├── sentiment/     ✅
└── correlation/   ✅
""")

✅ Correlation results saved!

market_sentiment/
├── raw/           ✅
├── processed/     ✅
├── sentiment/     ✅
└── correlation/   ✅

